In [22]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from IPython.display import Image

In [23]:
df = pd.read_csv('df_final.csv')
df.head()


,Цена,Дата публикации,Город,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,...,Month_Public,DayOfWeek_Public,Floor_Ratio,Is_First_Floor,Is_Last_Floor,Area_per_Room,Infrastructure_Score,Площадь_log,Цена_log,Цена_за_квадратный_метр_log
0,5964400,2025-09-22 13:37:50,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,24.000000,3.42,3.891820,15.601319,11.730126
1,5829810,2025-09-29 20:46:23,Киров,0,0,0,0,0,0,0,...,9,0,0.916667,0,0,39.000000,1.26,3.688879,15.578495,11.914940
2,6400900,2025-09-29 10:26:27,Киров,0,1,0,0,0,0,0,...,9,0,1.000000,0,1,17.333333,4.01,3.970292,15.671949,11.720714
3,5814900,2025-09-29 20:37:16,Киров,0,0,0,0,0,0,0,...,9,0,0.833333,0,0,39.000000,1.26,3.688879,15.575934,11.912379
4,6315750,2025-09-29 20:42:24,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,40.000000,1.26,3.713572,15.658557,11.969684


In [24]:
df.shape

(17509, 67)

In [25]:


# Построение гистограммы для распределения цен
fig_price_hist = px.histogram(
    df, 
    x='Цена', 
    nbins=100,
    title='Распределение Цен на Недвижимость',
    labels={'Цена': 'Цена (руб.)'},
    marginal='box' # Дополнительная информация о распределении
)

fig_price_hist.update_layout(
    title_x=0.5,
    xaxis_title='Цена (руб.)',
    yaxis_title='Количество Объектов'
)

fig_price_hist.show()

![1](./img/a.png)

## Анализ распределения цен


**Выводы:**
- Распределение цен **сильно смещено вправо** (положительная асимметрия), что характерно для ценовых данных недвижимости
- Это означает, что **подавляющее большинство квартир** находится в более низком и среднем ценовом сегменте
- Существует **"длинный хвост"** из очень дорогих, элитных объектов, которые являются скорее исключением
- Box plot показывает наличие **значительных выбросов** в верхней части распределения
- **Медиана заметно меньше среднего**, что подтверждает правостороннюю асимметрию



In [40]:
# Создание двух гистограмм для сравнения
fig = make_subplots(rows=1, cols=2, subplot_titles=('Исходное Распределение Цен', 'Логарифмированное Распределение Цен'))

# Исходное распределение
#fig.add_trace(go.Histogram(x=df['Цена'], nbinsx=100, name='Цена'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Цена_за_квадратный_метр'], nbinsx=100, name='Цена'), row=1, col=1)
# Логарифмированное распределение
#fig.add_trace(go.Histogram(x=df['Цена_log'], nbinsx=100, name='log(Цена + 1)'), row=1, col=2)

fig.add_trace(go.Histogram(x=df['Цена_за_квадратный_метр_log'], nbinsx=100, name='log(Цена + 1)'), row=1, col=2)

fig.update_layout(
    title_text='Сравнение Распределения Цены до и после Логарифмирования',
    title_x=0.5,
    showlegend=False,
    height=500
)
fig.update_xaxes(title_text='Цена (руб.)', row=1, col=1)
fig.update_xaxes(title_text='log(Цена + 1)', row=1, col=2)
fig.update_yaxes(title_text='Количество', row=1, col=1)
fig.update_yaxes(title_text='Количество', row=1, col=2)

fig.show()

![2](./img/b.png)

## Эффект логарифмического преобразования

**Наблюдения:**
- **Правая гистограмма выглядит гораздо более симметричной** и похожей на нормальное распределение
- Логарифмирование **существенно снижает асимметрию** распределения
- **"Длинный хвост" справа становится менее выраженным**

**Практические выводы:**
-  Для регрессионного моделирования цен **обязательно используем логарифмированную версию**
-  При интерпретации результатов модели нужно будет применить **обратное преобразование** (exp)


In [27]:
# Визуализация распределения цены за квадратный метр
fig_price_sqm_hist = px.histogram(
    df, 
    x='Цена_за_квадратный_метр', 
    nbins=100,
    title='Распределение Цены за Квадратный Метр',
    labels={'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    marginal='box'
)

fig_price_sqm_hist.update_layout(
    title_x=0.5,
    xaxis_title='Цена за м² (руб.)',
    yaxis_title='Количество Объектов'
)

fig_price_sqm_hist.show()

![3](./img/c.png)

## Анализ распределения цены за м²

**Наблюдения :**
- Распределение имеет **мультимодальный** характер - видно несколько пиков
- Это указывает на **неоднородность данных** - как минимум 2-3 различных рынка недвижимости


**Интерпретация пиков:**
1. **Первый пик** (низкие цены) - вероятно, регионы с низкой экономической активностью или эконом-класс
2. **Второй пик** (средние цены) - основной массовый сегмент, комфорт-класс
3. **Третий пик/хвост** (высокие цены) - бизнес/элитный класс, крупные города

**Статистические выводы:**
- Разница между медианой и средним говорит о наличии дорогих объектов, "тянущих" среднее вверх
- Диапазон между 10% и 90% квантилями показывает **широкий разброс** типичных цен



In [28]:
# Группировка данных по субъектам РФ и расчет медианной цены за м²
regional_prices = df.groupby('Субъект РФ').agg({
    'Цена_за_квадратный_метр': ['median', 'mean', 'std', 'count']
}).round(0)
regional_prices.columns = ['Медиана', 'Среднее', 'Ст.откл', 'Количество']
regional_prices = regional_prices.sort_values('Медиана', ascending=False).reset_index()

# Построение столбчатой диаграммы
fig_region_bar = px.bar(
    regional_prices,
    x='Субъект РФ',
    y='Медиана',
    title='Медианная Цена за Квадратный Метр по Субъектам РФ',
    labels={'Субъект РФ': 'Субъект РФ', 'Медиана': 'Медианная цена за м² (руб.)'},
    hover_data=['Среднее', 'Количество'],
    color='Медиана',
    color_continuous_scale='Viridis'
)

fig_region_bar.update_layout(title_x=0.5, xaxis_tickangle=-45)
fig_region_bar.show()


![4](./img/d.png)

## Медианная Цена за Квадратный Метр по Субъектам РФ

**Ключевые наблюдения:**
- Разница между самым дорогим и самым дешёвым регионом составляет **в несколько раз**
- Наблюдается **высокая вариативность цен** между субъектами РФ 
- Это **подтверждает критическую важность** географических признаков для модели
- Цветовая градация наглядно показывает **ценовые кластеры** регионов

**Гипотезы :**

 Регионы с высокими зарплатами → высокие цены  
 Регионы с растущим населением → растущие цены  
 Курортные регионы → большой + к цене  
 Промышленные регионы → средние/низкие цены

**Практические выводы:**
- Возможно, стоит создать **отдельные модели для регионов** с сильно различающимися ценами



In [29]:
# Для наглядности ограничимся городами с достаточным количеством объявлений
city_counts = df['Город'].value_counts()
top_cities = city_counts[city_counts > 50].index 

df_top_cities = df[df['Город'].isin(top_cities)]

# Сортировка городов по медианной цене для более наглядного графика
sorted_cities = df_top_cities.groupby('Город')['Цена_за_квадратный_метр'].median().sort_values(ascending=False).index

# Построение box plot
fig_city_box = px.box(
    df_top_cities,
    x='Город',
    y='Цена_за_квадратный_метр',
    title='Распределение Цен за Квадратный Метр по Городам',
    labels={'Город': 'Город', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    category_orders={'Город': sorted_cities} # Применяем сортировку
)

fig_city_box.update_layout(title_x=0.5)
fig_city_box.show()

![5](./img/e.png)


**Выводы:**

- В крупных городах разброс цен обычно больше, что отражает более сегментированный рынок (от эконом- до элитного класса).
- Наблюдается наличие выбросов в некоторых городах, что может указывать на объекты с уникальными характеристиками или ошибки в данных.

**Гипотезы:**
- Города с высокими медианными ценами могут быть центрами притяжения (рабочие места, образование, культура).
- Широкий разброс цен в некоторых городах может быть связан с неравномерным распределением объектов по районам с разной престижностью.


In [37]:
# Выбор числовых столбцов для анализа корреляции
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Расчет матрицы корреляции
corr_matrix = df[numeric_cols].corr()

# Для наглядности выберем только признаки с высокой корреляцией с ценой
corr_target = corr_matrix['Цена_log'].abs().sort_values(ascending=False)
top_corr_features = corr_target[corr_target > 0.1].index

# Тепловая карта для наиболее коррелирующих признаков
fig_heatmap = go.Figure(data=go.Heatmap(
                   z=df[top_corr_features].corr().values,
                   x=top_corr_features.tolist(),
                   y=top_corr_features.tolist(),
                   colorscale='RdBu',
                   zmin=-1,
                   zmax=1,
                   text=df[top_corr_features].corr().round(2).values,
                   texttemplate="%{text}",
                   textfont={"size": 10}
))

fig_heatmap.update_layout(
    title='Тепловая Карта Корреляций Ключевых Числовых Признаков (|r| > 0.1)',
    title_x=0.5,
    height=800,
    width=1000
)

fig_heatmap.show()

![6](./img/f.png)

## Анализ корреляций и мультиколлинеарности

**Ключевые наблюдения из корреляционного анализа:**

1. **Сильные предикторы цены (|r| > 0.5):**
   - Признаки с наиболее высокой корреляцией будут **основой для модели**

2. **Средние предикторы (0.2 < |r| < 0.5):**
   - Дают **дополнительную предсказательную силу**
   - Важны для тонкой настройки модели

3. **Слабые предикторы (|r| < 0.2):**
   - Могут быть исключены для **упрощения модели**
   - Но иногда важны в комбинации с другими признаками

**Типичные коррелирующие пары:**
- Площадь ↔ Количество комнат 
- Этаж ↔ Этажность_дома 
- Расстояния до разных объектов инфраструктуры




In [31]:
# Диаграмма рассеяния для площади и цены
sample_df = df.sample(min(5000, len(df)), random_state=42)

fig_scatter_area_price = px.scatter(
    sample_df,
    x='Площадь',
    y='Цена',
    color='Город',
    title='Зависимость Цены от Площади в Разрезе Городов',
    labels={'Площадь': 'Площадь (м²)', 'Цена': 'Цена (руб.)'},
    hover_data=['Цена_за_квадратный_метр', 'Rooms_Count'],
    opacity=0.6,
    trendline='ols'
)

fig_scatter_area_price.update_layout(
    title_x=0.5,
    height=600
)
fig_scatter_area_price.show()

# Корреляция площади и цены
print(f"Корреляция Площадь-Цена: {df['Площадь'].corr(df['Цена']):.3f}")


Корреляция Площадь-Цена: 0.711


![6](./img/g.png)

**Выводы:**
- Сильная положительная корреляция  - площадь ключевой предиктор
- Линия тренда показывает почти линейную зависимость
- Города образуют разные облака точек → разные цены за м²

**Для модели:** Площадь обязательно включить, рассмотреть взаимодействие с Городом


In [32]:
# Диаграмма рассеяния для расстояния до центра и цены за м²
sample_df = df.sample(min(5000, len(df)), random_state=42)

fig_scatter_dist_price = px.scatter(
    sample_df,
    x='dist_to_city_center',
    y='Цена_за_квадратный_метр',
    color='is_class_business',
    color_discrete_map={True: 'red', False: 'blue'},
    trendline='ols', 
    title='Зависимость Цены за м² от Расстояния до Центра',
    labels={
        'dist_to_city_center': 'Расстояние до центра (км)', 
        'Цена_за_квадратный_метр': 'Цена за м² (руб.)',
        'is_class_business': 'Бизнес-класс'
    },
    opacity=0.5
)

fig_scatter_dist_price.update_layout(
    title_x=0.5,
    height=600
)
fig_scatter_dist_price.show()

# Корреляция
print(f"Корреляция расстояние-цена: {df['dist_to_city_center'].corr(df['Цена_за_квадратный_метр']):.3f}")

Корреляция расстояние-цена: 0.038


![8](./img/l.png)

**Выводы:**
- Дальше от центра → дешевле
- Бизнес-класс стабильно дороже на любом расстоянии
- Тренд линейный, но с разбросом

**Для модели:** dist_to_city_center важен, учесть класс жилья

In [33]:
# Создаем временный DF для визуализации
material_df = df.melt(
    id_vars=['Цена_за_квадратный_метр'], 
    value_vars=['is_brick', 'is_monolith', 'is_panel'],
    var_name='Материал', 
    value_name='Есть признак'
)
material_df = material_df[material_df['Есть признак'] == True]
material_df['Материал'] = material_df['Материал'].replace({
    'is_brick': 'Кирпич', 
    'is_monolith': 'Монолит', 
    'is_panel': 'Панель'
})


# Box plot для материалов
fig_material = px.box(
    material_df, 
    x='Материал', 
    y='Цена_за_квадратный_метр',
    title='Влияние Материала Стен на Цену за м²',
    labels={'Материал': 'Материал стен', 'Цена_за_квадратный_метр': 'Цена за м² (руб.)'},
    color='Материал',
    points='outliers'
)
fig_material.update_layout(
    title_x=0.5,
    showlegend=False,
    height=500
)
fig_material.show()

![9](./img/m.png)

**Выводы:**
- Монолит > Кирпич > Панель по медианной цене
- Монолит: + ~15-25% к панели
- Значимое влияние на цену

**Для модели:** Материал стен важен



In [34]:
# Анализ влияния удобств
amenities_features = [
    'is_class_comfort', 'is_class_business', 'is_class_elite',
    'has_parking_underground', 'is_closed_yard', 'has_concierge'
]


# Создаем subplots
fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=[f.replace('_', ' ').title() for f in amenities_features]
)

row, col = 1, 1
for feature in amenities_features:
    fig.add_trace(
        go.Box(y=df[df[feature] == True]['Цена_за_квадратный_метр'], 
               name='Есть', marker_color='green'), 
        row=row, col=col
    )
    fig.add_trace(
        go.Box(y=df[df[feature] == False]['Цена_за_квадратный_метр'], 
               name='Нет', marker_color='gray'), 
        row=row, col=col
    )
    
    col += 1
    if col > 3:
        col = 1
        row += 1

fig.update_layout(
    title_text='Влияние Класса и Удобств на Цену за м²',
    title_x=0.5,
    showlegend=False,
    height=700
)
fig.show()

![10](./img/o.png)

**Выводы:**
- Все признаки дают + к цене 
- Наибольшее влияние: is_class_elite, is_class_business
- Удобства (паркинг, консьерж) также значимы

**Для модели:** Все эти признаки важны, оставить в модели


In [35]:
# Создание признака floor_ratio (если еще нет)
if 'floor_ratio' not in df.columns:
    df['floor_ratio'] = (df['Этаж'] / df['Этажность_дома']).fillna(0.5)
    print("✓ Создан признак floor_ratio")


# Диаграмма рассеяния
sample_df = df.sample(min(5000, len(df)), random_state=42)

fig_floor_ratio = px.scatter(
    sample_df,
    x='floor_ratio',
    y='Цена_за_квадратный_метр',
    title='Зависимость Цены за м² от Относительного Положения Этажа',
    labels={
        'floor_ratio': 'Относительное положение этажа (Этаж / Этажность)', 
        'Цена_за_квадратный_метр': 'Цена за м² (руб.)'
    },
    trendline='lowess',
    opacity=0.4
)

fig_floor_ratio.update_layout(
    title_x=0.5,
    height=600
)
fig_floor_ratio.show()

✓ Создан признак floor_ratio


![11](./img/p.png)

**Выводы:**
- Первый этаж дешевле средних
- Средние этажи (0.3-0.7) оптимальны по цене
- Последний этаж: зависит от типа дома

**Для модели:** floor_ratio полезен, можно добавить Is_First_Floor, Is_Last_Floor


In [36]:
# Агрегация данных на уровне города
city_agg = df.groupby('Город').agg(
    median_price_sqm=('Цена_за_квадратный_метр', 'median'),
    avg_salary=('Cредняя зп в городе, тыс руб (2025)', 'mean'),
    population=('2015 население', 'mean'),
    population_dynamics=('Динамика населения за 10 лет', 'mean'),
    count=('Цена', 'count')
).reset_index()


# Диаграмма рассеяния
fig_macro = px.scatter(
    city_agg,
    x='avg_salary',
    y='median_price_sqm',
    size='population',
    text='Город',
    title='Связь Средней Зарплаты и Медианной Цены за м² по Городам',
    labels={
        'avg_salary': 'Средняя ЗП в городе (тыс. руб.)', 
        'median_price_sqm': 'Медианная цена за м² (руб.)',
        'population': 'Население'
    },
    hover_data=['count', 'population_dynamics'],
    trendline='ols'
)

fig_macro.update_traces(textposition='top center', textfont_size=9)
fig_macro.update_layout(
    title_x=0.5, 
    height=650
)
fig_macro.show()

![13](./img/x.png)

**Выводы:**
- Средняя ЗП сильно коррелирует с ценами 
- Тренд положительный: больше зарплата → дороже жилье
- Размер населения влияет слабее

**Для модели:** ЗП и макропоказатели важны, включить в модель


## 📝 Краткие выводы EDA

### Целевая переменная:
- Распределение цен сильно скошено вправо → **использовать log(Цена)**
- Мультимодальное распределение → неоднородный рынок

### Ключевые факторы влияния на цену:

**1. Географические (сильное влияние):**
- Регион и город - критически важны
- Расстояние до центра - отрицательная корреляция
- Средняя ЗП региона - сильная корреляция

**2. Характеристики объекта (сильное влияние):**
- Площадь - основной предиктор 
- Материал: Монолит > Кирпич > Панель
- Класс жилья: + 10-50%

**3. Дополнительные факторы:**
- Удобства (паркинг, консьерж) - + к цене
- Этаж - слабое влияние, первый дешевле
- Макропоказатели - умеренное влияние

### Рекомендации для модели:
- Использовать log-преобразование целевой  
- Обязательные признаки: Площадь, Регион/Город, Класс, Материал  
- Проверить мультиколлинеарность  

